# Diffusion counterfactual parameters

Only the parameters that differ from the FastDiME reference (`FastDiME_Med/CFgenerating/config_cf.py`).
For each: what it does, what we chose, why, and the measured effect. Nothing is run here; it reads the calibration and sweep CSVs.

In [ ]:
import os, json
import pandas as pd
R = os.environ.get("THESIS_RESULTS", "/work3/s251710/thesis_results") + "/CF_levels"
pd.options.display.float_format = "{:.3f}".format

## 1. Overview

| Parameter | Reference | Ours | What it does | Why |
|---|---|---|---|---|
| tau (`cf_from_timestep`) | 400 | **150** | noise added before denoising starts; cost is linear in it | 50 collapses the levels, 200 doubles cost (§2) |
| gradient scale | 100 | **150** | multiplies the classifier gradient each step | largest value before the levels differ in sharpness (§3) |
| Denoised_L1 weight | 200 | **10** | pull back towards the original image | **not yet justified, see §4** |
| inpainting mask | on | **off** (`FastDiME-woM`) | confines the edit to a region | we do not hunt shortcuts; edits need not be local |
| target | hard class flip | **soft probability, 3 levels** | BCE minimum sits at the target | graded intensity; one full trajectory per level (§5) |
| classifier | time-conditioned UNet encoder | **plain DenseNet C0** | | forced; loss is arithmetically identical |

## 2. tau: more steps separate the levels, at linear cost

In [ ]:
cal = pd.read_csv(f"{R}/calibration.csv")
cal = cal[(cal.scale == 100) & (cal.disease == "cardiomegaly")]
cal = cal.assign(secs_per_img=cal.sec_per_img)[["tau","sep_12","sep_23","dL1_12","L1_final","reach_3","secs_per_img"]]
cal.rename(columns={"sep_12":"stop-t gap L1→L2","sep_23":"stop-t gap L2→L3","dL1_12":"pixel dist L1→L2",
                    "L1_final":"edit size","reach_3":"reach of top level"})

Cardiomegaly, scale 100. At tau 50 the gap between the first two levels is a few steps and they differ by ~0.001 in pixel
distance: the same image with two labels. A larger tau separates them and lets the top level be reached, but the edit grows
and so does the cost. 150 sits between the 100 and 200 rows (only 50/100/200 were measured).

## 3. Scale: accuracy against level-to-level sharpness

In [ ]:
sw = pd.read_csv(f"{R}/soft_sweep_stage1.csv")
sw[["disease","scale","mean_err","spread","sharp_range","l1_total"]]

`mean_err` is the distance of the final probabilities from the three targets. Raising scale shrinks it on effusion
(0.065 → 0.029) but `sharp_range`, the sharpness difference between levels, grows from 0.003 to 0.089. A downstream model could then
tell levels apart by blur, which the per-level trajectories were designed to prevent. Cardiomegaly already has ~0 bias at 150.
n = 8 images per cell, so treat the bias columns as indicative.

## 4. Denoised_L1 weight: OPEN

Reference 200, ours 10 (`W_DENOISED_L1` in `cf_generation_final.py`). Lower means a weaker pull back to the original and so larger edits.
No sweep or note supports 10. Fill the table below from an ablation at fixed tau=150, scale=150 before using this in the thesis.

| weight | mean_err | l1_total | sharp_range |
|---|---|---|---|
| 10 (current) | | | |
| 50 | | | |
| 200 (reference) | | | |

## 5. Soft targets: three equally spaced probabilities

In [ ]:
lv = json.load(open(f"{R}/effusion/levels.json"))
pd.DataFrame({"level": lv["level_names"], "to_positive": lv["to_positive"], "to_negative": lv["to_negative"]})

Effusion. Endpoints are the 5th and 95th percentile of correctly classified cases of the target class, so a fixed top level
cannot be unreachable for a weak disease; the middle level is the midpoint. Each level is a full tau→1 trajectory with the same seed,
so all three are fully denoised and equally sharp (0.6% range, against 1.6% and 2.6% for the snapshot designs tried first).